In [1]:
# Télécharge, via l'API transport.data.gouv.fr (même routine que
# scripts/rafraichir_gtfs.py : recuperer_datasets_public_transit +
# resultat_pour_page_url, pas un simple re-téléchargement de
# ressource_url en cache — resultat_pour_page_url ré-interroge le PAN
# pour retrouver la ressource *actuelle* du dataset, potentiellement
# différente si la ressource a changé d'URL depuis le dernier
# enregistrement de provenance), tous les GTFS déjà associés à un
# dataset dans data/gtfs_sources.json — dans data/GTFS_temp/, PAS
# data/GTFS/ (ne remplace rien automatiquement, à l'écart pour
# inspection/comparaison manuelle avant de basculer si besoin).
#
# Les GTFS de gtfs_sources.json sans page_url (associés seulement pour
# leur académie/zone, cf. src.vacances_scolaires — jamais liés à un
# dataset PAN) sont ignorés : rien à télécharger pour eux ici.

import os

from src.transport_data_gouv import (
    charger_provenance,
    recuperer_datasets_public_transit,
    resultat_pour_page_url,
    telecharger_gtfs,
)

DOSSIER_TEMP = os.path.join("data", "GTFS_temp")
os.makedirs(DOSSIER_TEMP, exist_ok=True)

provenance = charger_provenance()
a_telecharger = {f: info for f, info in provenance.items() if info.get("page_url")}
print(f"{len(a_telecharger)} GTFS liés à transport.data.gouv.fr à télécharger dans {DOSSIER_TEMP}\n")

print("Récupération du catalogue transport.data.gouv.fr...")
datasets = recuperer_datasets_public_transit()

reussis, introuvables, echecs = [], [], []
for nom_fichier, info in sorted(a_telecharger.items()):
    resultat = resultat_pour_page_url(info["page_url"], info.get("ressource_url"), datasets)
    if resultat is None:
        print(f"⚠ {nom_fichier} : dataset introuvable sur transport.data.gouv.fr (page supprimée/déplacée ?)")
        introuvables.append(nom_fichier)
        continue
    try:
        contenu = telecharger_gtfs(resultat)
        chemin_cible = os.path.join(DOSSIER_TEMP, nom_fichier)
        with open(chemin_cible, "wb") as f:
            f.write(contenu)
        print(f"✓ {nom_fichier} ({len(contenu) / 1e6:.1f} Mo, màj {resultat['ressource_maj']})")
        reussis.append(nom_fichier)
    except Exception as e:
        print(f"✗ {nom_fichier} : {type(e).__name__}: {e}")
        echecs.append(nom_fichier)

print(f"\n{len(reussis)} réussi(s), {len(introuvables)} introuvable(s), {len(echecs)} échec(s)")
if introuvables:
    print("Introuvables :", introuvables)
if echecs:
    print("Échecs :", echecs)

54 GTFS liés à transport.data.gouv.fr à télécharger dans data/GTFS_temp

Récupération du catalogue transport.data.gouv.fr...


✓ Albi_libea-reseau-urbain.zip (0.0 Mo, màj 2026-06-22T08:47:55.785000Z)


✓ Ales_gtfs-is-20260704.zip (0.3 Mo, màj 2026-06-22T13:25:53.192000Z)


✓ Amiens_gtfs-fusion-20260721-1000.zip (3.0 Mo, màj 2026-07-22T09:10:44.162000Z)


✓ Angers_gtfs.zip (3.0 Mo, màj 2026-07-30T00:00:46.000000Z)


...

✓ Toulouse_tisseo_gtfs_v2_r5py.zip (18.7 Mo, màj 2026-08-11T03:51:50.265457Z)


✓ Tours_filbleu_gtfs14.zip (10.5 Mo, màj 2026-08-08T06:18:14.925161Z)


✓ Valence_gtfs-citea-20260517-20260628.zip (11.5 Mo, màj 2026-05-18T07:14:00.640000Z)


✓ Valenciennes_google-transit.zip (1.3 Mo, màj 2026-07-23T10:12:16.530000Z)


✓ Vannes_GTFS.zip (8.3 Mo, màj 2026-07-28T15:22:21.439000Z)

53 réussi(s), 0 introuvable(s), 1 échec(s)
Échecs : ['Lyon_GTFS_TCL.zip']


In [2]:
# Calcule, pour chaque GTFS téléchargé dans data/GTFS_temp/ (cellule
# précédente), sa période de validité (date_debut/date_fin) et sa date
# JOB (dernier mardi/jeudi hors vacances scolaires de l'académie si
# connue, cf. src.info_reseau.dates_service) — écrit dans
# data/GTFS_temp/gtfs_sources_temp.json pour comparer avant de basculer
# ces fichiers vers data/GTFS/ (rien n'est remplacé automatiquement ici).

import json

from src.info_reseau import dates_service
from src.utils import charger_gtfs
from src.vacances_scolaires import departement_academie_zone_pour_feed

fichiers = sorted(f for f in os.listdir(DOSSIER_TEMP) if f.lower().endswith(".zip"))
print(f"{len(fichiers)} GTFS à analyser dans {DOSSIER_TEMP}\n")

resultats = {}
for nom_fichier in fichiers:
    chemin = os.path.join(DOSSIER_TEMP, nom_fichier)
    try:
        feed = charger_gtfs(chemin)
    except Exception as e:
        print(f"✗ {nom_fichier} : impossible de charger ({type(e).__name__}: {e})")
        continue

    try:
        _, academie, _ = departement_academie_zone_pour_feed(feed)
    except Exception:
        academie = None

    try:
        _, date_debut, date_fin, date_job = dates_service(feed, academie=academie)
    except Exception as e:
        print(f"✗ {nom_fichier} : échec dates_service ({type(e).__name__}: {e})")
        continue

    resultats[nom_fichier] = {
        "academie": academie,
        "date_debut": date_debut,
        "date_fin": date_fin,
        "date_JOB": date_job,
    }
    print(f"✓ {nom_fichier} : {date_debut} -> {date_fin} (JOB {date_job})")

chemin_json = os.path.join(DOSSIER_TEMP, "gtfs_sources_temp.json")
with open(chemin_json, "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2, sort_keys=True)

print(f"\n{len(resultats)}/{len(fichiers)} GTFS analysé(s) — écrit dans {chemin_json}")


52 GTFS à analyser dans data/GTFS_temp

Chargement du fichier GTFS : data/GTFS_temp/Albi_libea-reseau-urbain.zip
✓ GTFS chargé avec succès


✓ Albi_libea-reseau-urbain.zip : 20260601 -> 20261231 (JOB 20261217)
Chargement du fichier GTFS : data/GTFS_temp/Ales_gtfs-is-20260704.zip
✓ GTFS chargé avec succès


✓ Ales_gtfs-is-20260704.zip : 20260504 -> 20260703 (JOB 20260702)
Chargement du fichier GTFS : data/GTFS_temp/Amiens_gtfs-fusion-20260721-1000.zip


✓ GTFS chargé avec succès


✓ Amiens_gtfs-fusion-20260721-1000.zip : 20260715 -> 20270112 (JOB 20270112)
Chargement du fichier GTFS : data/GTFS_temp/Angers_gtfs.zip


...

✓ GTFS chargé avec succès
✓ Valence_gtfs-citea-20260517-20260628.zip : 20260518 -> 20260626 (JOB 20260625)
Chargement du fichier GTFS : data/GTFS_temp/Valenciennes_google-transit.zip


✓ GTFS chargé avec succès
✓ Valenciennes_google-transit.zip : 20260706 -> 20260831 (JOB 20260827)


Chargement du fichier GTFS : data/GTFS_temp/Vannes_GTFS.zip


✓ GTFS chargé avec succès


✓ Vannes_GTFS.zip : 20260622 -> 20261116 (JOB 20261112)

52/52 GTFS analysé(s) — écrit dans data/GTFS_temp/gtfs_sources_temp.json
